# FUNSD benchmark: four ways to read a page

Runs the handwritten route on `nielsr/funsd` test pages in four modes and scores all of them against the labelled **answers**:

| Mode | What runs |
| --- | --- |
| `page_read` | Qwen3-VL-4B reads the whole page in one call (the PRD v1.5 route) |
| `page_read+crop` | the same, then OpenCV box snapping and a cropped re-read of low-confidence and ID fields |
| `blocks` | OpenCV groups the text into blocks (up to 30), each block is cropped and the model reads it as plain OCR |
| `strips` | the page is cut into 6 horizontal strips (with overlap), the model extracts fields from each strip, results merged |

Set **Internet: ON** and **GPU T4**, then run the four code cells in order. Change `PAGES` in cell 3 (the test split has 50 pages; start with 5, because `blocks` and `strips` make many model calls per page).

In [ ]:
# 1. Get the code, install what is missing (first run takes a few minutes)
import os, subprocess
os.chdir("/kaggle/working")
REPO = "/kaggle/working/DOC_Extractor_QUAPT"
BRANCH = "feat/page-read"
URL = "https://github.com/dharicharan/DOC_Extractor_QUAPT.git"
if not os.path.isdir(REPO + "/.git"):
    r = subprocess.run(["git", "clone", "-b", BRANCH, URL, REPO], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("Clone failed (Internet ON?): " + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH])
    subprocess.run(["git", "-C", REPO, "checkout", "-B", BRANCH, "origin/" + BRANCH])
print("commit:", subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
try:
    import transformers, qwen_vl_utils, bitsandbytes, cv2, datasets, doc_extractor
except Exception:
    get_ipython().system("pip install -q uv")
    get_ipython().system(f"cd {REPO} && uv pip install --system -r requirements.txt && uv pip install --system -e .")
    get_ipython().system("uv pip install --system qwen-vl-utils transformers accelerate bitsandbytes datasets")
os.environ.setdefault("DOC_MAX_IMAGE_PIXELS", "2000000")

In [ ]:
# 2. Helpers: gold answers from the FUNSD labels, the readers (fields, blocks, strips), scoring
import re, io
import numpy as np, cv2
from PIL import Image
from difflib import SequenceMatcher
from doc_extractor.crop_verify import _as_box, _leaves, crop_region
from doc_extractor.vlm_grounding import _null_word_to_none


def norm(s):
    return re.sub(r"[^0-9a-z]", "", str(s).lower())


def words_of(s):
    return re.findall(r"[0-9a-z]+", str(s).lower())


def gold_answers(words, boxes, tags):
    """ANSWER entities (tags 5 = B-ANSWER, 6 = I-ANSWER): consecutive words joined, boxes unioned."""
    out, cur = [], None
    for w, b, t in zip(words, boxes, tags):
        if t == 5 or (t == 6 and cur is None):
            if cur:
                out.append(cur)
            cur = {"w": [w], "b": list(b)}
        elif t == 6:
            cur["w"].append(w)
            cur["b"] = [min(cur["b"][0], b[0]), min(cur["b"][1], b[1]), max(cur["b"][2], b[2]), max(cur["b"][3], b[3])]
        else:
            if cur:
                out.append(cur)
            cur = None
    if cur:
        out.append(cur)
    return [{"text": " ".join(a["w"]), "box": tuple(map(float, a["b"]))} for a in out if norm(" ".join(a["w"]))]


def same_text(p, g):
    p, g = norm(p), norm(g)
    if not p or not g:
        return False
    return p == g or (len(g) >= 4 and g in p) or SequenceMatcher(None, p, g).ratio() >= 0.8


def iou(a, b):
    if not a or not b:
        return 0.0
    inter = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def covered(gbox, b):
    """Share of the gold answer's box that lies inside box b."""
    if not b:
        return 0.0
    inter = max(0, min(gbox[2], b[2]) - max(gbox[0], b[0])) * max(0, min(gbox[3], b[3]) - max(gbox[1], b[1]))
    area = (gbox[2] - gbox[0]) * (gbox[3] - gbox[1])
    return inter / area if area > 0 else 0.0


def png_bytes(img):
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="PNG")
    return buf.getvalue()


# ---- reader A: the model's field JSON (page_read, page_read+crop) -> items
def predicted(raw):
    out = []
    for path, leaf in _leaves(raw):
        v = _null_word_to_none(leaf.get("value"))
        if v is None or isinstance(v, (dict, list)) or not norm(v):
            continue
        out.append({"text": str(v), "box": _as_box(leaf.get("bbox")), "flagged": leaf.get("crop_verified") is False})
    return out


# ---- reader B: OpenCV text blocks, each cropped and read by the model as plain OCR
def find_blocks(img, max_blocks=30):
    rgb = np.asarray(img.convert("RGB"))
    h, w = rgb.shape[:2]
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, max(15, (min(h, w) // 40) | 1), 15)
    hl = cv2.morphologyEx(binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (max(25, w // 12), 1)))
    vl = cv2.morphologyEx(binary, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(25, h // 12))))
    ink = cv2.subtract(binary, cv2.dilate(cv2.bitwise_or(hl, vl), np.ones((3, 3), np.uint8)))      # form rules removed
    joined = cv2.dilate(ink, cv2.getStructuringElement(cv2.MORPH_RECT, (max(15, int(w * 0.06)), max(5, int(h * 0.016)))))
    contours, _ = cv2.findContours(joined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rects = [cv2.boundingRect(c) for c in contours]
    rects = [r for r in rects if r[2] >= 0.03 * w and r[3] >= 0.008 * h and r[2] * r[3] < 0.6 * w * h]
    rects = sorted(rects, key=lambda r: r[2] * r[3], reverse=True)[:max_blocks]        # keep the biggest blocks
    rects = sorted(rects, key=lambda r: (r[1] // max(1, h // 40), r[0]))               # reading order
    return [(1000.0 * x / w, 1000.0 * y / h, 1000.0 * (x + bw) / w, 1000.0 * (y + bh) / h) for x, y, bw, bh in rects]


OCR_SYSTEM = ("You are an OCR engine. Output only the text in the image, exactly as written, one line per line of "
              "text, in reading order. No explanation, no quotes.")


def read_blocks(extractor, img):
    items = []
    for box in find_blocks(img):
        crop = crop_region(img, box, pad=0.12)
        cw, ch = crop.size
        scale = max(1.0, 64.0 / min(cw, ch), min(2.5, 1000.0 / cw))
        if scale > 1.0:
            crop = crop.resize((int(cw * scale), int(ch * scale)), Image.LANCZOS)
        text = extractor._generate_from_image(png_bytes(crop), OCR_SYSTEM, "Transcribe all the text.", 256).strip()
        if norm(text):
            items.append({"text": text, "box": box, "flagged": False})
    return items


# ---- reader C: horizontal strips, each read by the model like a small page, results merged
def read_strips(extractor, img, prompt, n=6, overlap=0.15):
    W, H = img.size
    step = H / n
    strip_prompt = prompt + (" NOTE: this image is one horizontal strip of a larger page. A field may be cut at the "
                             "top or bottom edge; read only complete entries.")
    merged = []
    for k in range(n):
        y0, y1 = max(0, int(k * step - overlap * step)), min(H, int((k + 1) * step + overlap * step))
        strip = img.crop((0, y0, W, y1))
        s = max(1.0, 1600.0 / W)
        strip = strip.resize((int(W * s), int((y1 - y0) * s)), Image.LANCZOS)
        raw, _ = extractor.extract_page_image(png_bytes(strip), None, system_prompt=strip_prompt)
        for path, leaf in _leaves(raw):
            v = _null_word_to_none(leaf.get("value"))
            if v is None or isinstance(v, (dict, list)) or not norm(v):
                continue
            b = _as_box(leaf.get("bbox"))
            if b:   # strip grid -> page grid
                b = (b[0], (y0 + b[1] / 1000.0 * (y1 - y0)) / H * 1000.0, b[2], (y0 + b[3] / 1000.0 * (y1 - y0)) / H * 1000.0)
            merged.append({"text": str(v), "box": b, "flagged": False})
    out = []
    for it in merged:    # the overlap reads some entries twice
        if any(norm(it["text"]) == norm(o["text"]) and (not it["box"] or not o["box"] or iou(it["box"], o["box"]) > 0.3) for o in out):
            continue
        out.append(it)
    return out


# ---- scoring (the same for every mode)
def score(items, golds):
    pairs = sorted(((SequenceMatcher(None, norm(p["text"]), norm(g["text"])).ratio(), i, j)
                    for i, p in enumerate(items) for j, g in enumerate(golds) if same_text(p["text"], g["text"])), reverse=True)
    up, ug, m = set(), set(), []
    for _, i, j in pairs:
        if i not in up and j not in ug:
            up.add(i)
            ug.add(j)
            m.append((items[i], golds[j]))
    ious = [iou(p["box"], g["box"]) for p, g in m]
    found = {w for it in items for w in words_of(it["text"])}
    gold_words = [w for g in golds for w in words_of(g["text"])]
    return {"gold": len(golds), "pred": len(items), "matched": len(m), "box_hits": sum(v >= 0.5 for v in ious),
            "iou": sum(ious), "flagged": sum(p["flagged"] for p in items),
            "flagged_wrong": sum(p["flagged"] and k not in up for k, p in enumerate(items)),
            "gold_words": len(gold_words), "words_found": sum(w in found for w in gold_words),
            "covered": sum(any(covered(g["box"], it["box"]) >= 0.5 for it in items) for g in golds)}

In [ ]:
# 3. Run every mode on FUNSD test pages (the model loads once)
import tempfile, time, json
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from doc_extractor import crop_verify
from doc_extractor.adapters import VLMAdapter
from doc_extractor.routing import DEFAULT_MODEL, DocumentCategory, RoutedExtractor, build_generic_prompt

PAGES = 5           # start small: strips and blocks make many model calls per page. The test split has 50 pages.
MODES = ["page_read", "page_read+crop", "blocks", "strips"]    # remove names here to skip a mode

ds = load_dataset("nielsr/funsd", split="test")
rows = [ds[i] for i in range(min(PAGES, len(ds)))]
extractor = RoutedExtractor().get_extractor(DEFAULT_MODEL)
adapter = VLMAdapter(extractor)
adapter.crop_verify_allowed = True
prompt = build_generic_prompt(DocumentCategory.HANDWRITTEN)
tmp = Path(tempfile.mkdtemp())
paths = []
for r in rows:
    p = tmp / f"funsd_{r['id']}.png"
    r["image"].convert("RGB").save(p)
    paths.append(p)


def run_mode(mode, img, path):
    """Returns (items, model_calls) for one page."""
    if mode in ("page_read", "page_read+crop"):
        os.environ["DOC_CROP_VERIFY"] = "1" if mode.endswith("crop") else "0"
        raw, _, meta = adapter.extract(str(path), prompt)
        return predicted(raw), 1 + (meta.get("crop_verify") or {}).get("verified", 0)
    if mode == "blocks":
        items = read_blocks(extractor, img)
        return items, len(items)
    return read_strips(extractor, img, prompt), 6


results = {}
for mode in MODES:
    per = []
    for r, p in zip(rows, paths):
        golds = gold_answers(r["words"], r["bboxes"], r["ner_tags"])
        t0 = time.perf_counter()
        try:
            items, calls = run_mode(mode, r["image"].convert("RGB"), p)
        except Exception as e:
            print(mode, r["id"], "FAILED", type(e).__name__, e)
            items, calls = [], 0
        s = score(items, golds)
        s.update(sec=time.perf_counter() - t0, calls=calls)
        per.append(s)
        print(f"[{mode}] page {r['id']}: {s['words_found']}/{s['gold_words']} answer words, {s['covered']}/{s['gold']} boxes covered, {calls} calls, {s['sec']:.0f}s")
    results[mode] = per
    crop_verify.begin_run()

In [ ]:
# 4. Result table
def summary(mode, per):
    t = {k: sum(p[k] for p in per) for k in per[0]}
    field_mode = mode != "blocks"      # blocks return plain text, not fields, so field-level numbers do not apply
    return {
        "answer words found": round(t["words_found"] / max(t["gold_words"], 1), 3),
        "answer boxes covered": round(t["covered"] / max(t["gold"], 1), 3),
        "answers matched as a field": round(t["matched"] / t["gold"], 3) if field_mode else "-",
        "value precision": round(t["matched"] / max(t["pred"], 1), 3) if field_mode else "-",
        "box hit rate (IoU>=0.5)": round(t["box_hits"] / max(t["matched"], 1), 3) if field_mode else "-",
        "fields flagged by crop check": t["flagged"] if mode == "page_read+crop" else "-",
        "flag precision (flagged were wrong)": round(t["flagged_wrong"] / t["flagged"], 3) if mode == "page_read+crop" and t["flagged"] else "-",
        "model calls per page": round(t["calls"] / len(per), 1),
        "seconds per page": round(t["sec"] / len(per), 1),
    }


table = pd.DataFrame({m: summary(m, per) for m, per in results.items()})
display(table)
json.dump(results, open("/kaggle/working/funsd_results.json", "w"), indent=1)
print("Saved /kaggle/working/funsd_results.json")

## Reading the table
- **answer words found:** share of the labelled answer words that appear anywhere in the mode's output. This is the fair number to compare all four modes.
- **answer boxes covered:** share of labelled answers whose box lies inside one of the mode's boxes.
- **answers matched as a field, value precision, box hit rate:** only for the modes that return fields (not `blocks`).
- **flag precision:** of the fields the crop check flagged, the share that really were wrong (`page_read+crop` only).
- **calls and seconds per page:** the cost. More calls means slower.
- FUNSD pages are typed or printed forms, not handwriting, so use it to compare recall, boxes and cost. Judge handwriting on your own forms.